# 🦺 Sistema de Detección de EPP en Construcción con YOLOv8

**Dataset base:** SH17 Dataset for PPE Detection de Kaggle  
**Adaptación:** filtrado automático a clases útiles para construcción.

Clases finales:
```text
Person, Ear, Glasses, Helmet, Face, Gloves, Hands, Head, Shoes, Safety-vest
```

Antes de ejecutar: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4`.


In [ ]:
# ============================================================
# 0. Instalar dependencias
# ============================================================
!pip install -q ultralytics kaggle opencv-python-headless scikit-learn scikit-image matplotlib pandas pyyaml
print("✅ Dependencias instaladas")


In [ ]:
# ============================================================
# 1. Imports y directorios
# ============================================================
import os, json, zipfile, shutil, random, time
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from skimage.feature import hog

import torch

BASE_DIR = Path("/content/epp_sh17_construction_project")
DATA_DIR = BASE_DIR / "datasets"
RAW_DIR = DATA_DIR / "sh17_raw"
FILTERED_DIR = DATA_DIR / "sh17_construction_filtered"
RUNS_DIR = BASE_DIR / "runs"
FIG_DIR = BASE_DIR / "figures"
OUT_DIR = BASE_DIR / "outputs"

for d in [BASE_DIR, DATA_DIR, RAW_DIR, FILTERED_DIR, RUNS_DIR, FIG_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Directorios listos")
print("BASE_DIR:", BASE_DIR)


In [ ]:
# ============================================================
# 1. Imports y directorios
# ============================================================
import os, json, zipfile, shutil, random, time
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from skimage.feature import hog

import torch

BASE_DIR = Path("/content/epp_sh17_construction_project")
DATA_DIR = BASE_DIR / "datasets"
RAW_DIR = DATA_DIR / "sh17_raw"
FILTERED_DIR = DATA_DIR / "sh17_construction_filtered"
RUNS_DIR = BASE_DIR / "runs"
FIG_DIR = BASE_DIR / "figures"
OUT_DIR = BASE_DIR / "outputs"

for d in [BASE_DIR, DATA_DIR, RAW_DIR, FILTERED_DIR, RUNS_DIR, FIG_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Directorios listos")
print("BASE_DIR:", BASE_DIR)


In [ ]:
# ============================================================
# 2. Verificar GPU
# ============================================================
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = 0
else:
    print("⚠️ No hay GPU. Activa GPU T4.")
    DEVICE = "cpu"

!nvidia-smi | head -20



## 3. Autenticación Kaggle

Edita la siguiente celda y coloca tu usuario y API key de Kaggle.

Para obtenerlos:
1. Kaggle > perfil > Account.
2. Sección API > Create New Token.
3. Abre `kaggle.json`.
4. Copia `"username"` y `"key"`.


In [ ]:
# ============================================================
# 3. Autenticación Kaggle
# ============================================================
import os

# REEMPLAZA ESTOS DOS VALORES
os.environ["KAGGLE_USERNAME"] = "rodrigoabelrodriguez"
os.environ["KAGGLE_KEY"] = "KGAT_6525bfab65ad2091caa12f3f93ff81cb"

if os.environ["KAGGLE_USERNAME"] == "TU_USUARIO_DE_KAGGLE":
    raise ValueError("Debes reemplazar TU_USUARIO_DE_KAGGLE por tu usuario real de Kaggle.")

if os.environ["KAGGLE_KEY"] == "TU_API_KEY_DE_KAGGLE":
    raise ValueError("Debes reemplazar TU_API_KEY_DE_KAGGLE por tu API key real de Kaggle.")

print("✅ Kaggle configurado con variables de entorno")
print("Usuario:", os.environ["KAGGLE_USERNAME"])

In [ ]:
# ============================================================
# 4. Descargar SH17 desde Kaggle
# ============================================================
DATASET_SLUG = "mugheesahmad/sh17-dataset-for-ppe-detection"

print("Descargando dataset:", DATASET_SLUG)
!kaggle datasets download -d {DATASET_SLUG} -p {RAW_DIR} --unzip

print("✅ Descarga finalizada")
!find {RAW_DIR} -maxdepth 3 -type f | head -40

In [ ]:

# ============================================================
# 5. Configuracion de clases SH17 y filtro para construccion
# ============================================================

# IMPORTANTE:
# El orden de clases del dataset SH17 descargado puede no coincidir con listas
# encontradas en internet o notebooks previos. Por eso leemos los nombres reales
# desde el YAML/archivo de clases incluido en la descarga y remapeamos por nombre,
# no por indices escritos a mano.

def normalize_class_name(name):
    return str(name).strip().lower().replace("_", "-").replace(" ", "-")

CLASS_ALIASES = {
    "person": "Person",
    "head": "Head",
    "face": "Face",
    "glasses": "Glasses",
    "face-mask-medical": "Face-mask-medical",
    "face-mask": "Face-mask-medical",
    "face-guard": "Face-guard",
    "ear": "Ear",
    "earmuffs": "Earmuffs",
    "ear-mufs": "Earmuffs",
    "ear-muffs": "Earmuffs",
    "hands": "Hands",
    "hand": "Hands",
    "gloves": "Gloves",
    "glove": "Gloves",
    "foot": "Foot",
    "shoes": "Shoes",
    "shoe": "Shoes",
    "safety-vest": "Safety-vest",
    "vest": "Safety-vest",
    "tools": "Tools",
    "tool": "Tools",
    "helmet": "Helmet",
    "medical-suit": "Medical-suit",
    "safety-suit": "Safety-suit",
}

# Este orden preserva los IDs semanticos que ya usa el proyecto local:
# 0 Person, 1 Ear, 2 Glasses, 3 Helmet, 4 Face, 5 Gloves,
# 6 Hands, 7 Head, 8 Shoes. Agregamos chaleco como ID 9.
CONSTRUCTION_NAMES = [
    "Person",
    "Ear",
    "Glasses",
    "Helmet",
    "Face",
    "Gloves",
    "Hands",
    "Head",
    "Shoes",
    "Safety-vest",
]

REQUIRED_NAMES = set(CONSTRUCTION_NAMES)


def names_from_yaml_value(value):
    if isinstance(value, dict):
        return [value[k] for k in sorted(value, key=lambda x: int(x))]
    if isinstance(value, list):
        return value
    return None


def find_original_names(root):
    root = Path(root)

    yaml_candidates = sorted(
        [p for p in root.rglob("*.yaml") if p.is_file()] +
        [p for p in root.rglob("*.yml") if p.is_file()]
    )
    for yaml_path in yaml_candidates:
        try:
            with open(yaml_path, "r", encoding="utf-8", errors="ignore") as f:
                data = yaml.safe_load(f)
        except Exception:
            continue

        if isinstance(data, dict) and "names" in data:
            names = names_from_yaml_value(data["names"])
            if names:
                print("✅ Nombres de clases leidos desde:", yaml_path)
                return [CLASS_ALIASES.get(normalize_class_name(n), str(n).strip()) for n in names]

    for file_name in ["classes.txt", "obj.names"]:
        for txt_path in sorted(root.rglob(file_name)):
            names = [line.strip() for line in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
            if names:
                print("✅ Nombres de clases leidos desde:", txt_path)
                return [CLASS_ALIASES.get(normalize_class_name(n), str(n).strip()) for n in names]

    raise RuntimeError(
        "No se encontraron nombres de clases en el dataset descargado. "
        "Revisa si existe data.yaml, classes.txt u obj.names dentro de RAW_DIR."
    )


ORIGINAL_NAMES = find_original_names(RAW_DIR)
RAW_NAME_TO_ID = {normalize_class_name(name): idx for idx, name in enumerate(ORIGINAL_NAMES)}

missing_names = [name for name in CONSTRUCTION_NAMES if normalize_class_name(name) not in RAW_NAME_TO_ID]
if missing_names:
    raise RuntimeError(
        "Faltan estas clases en el dataset original: " + ", ".join(missing_names) +
        "\nClases encontradas: " + ", ".join(ORIGINAL_NAMES)
    )

OLD_TO_NEW = {
    RAW_NAME_TO_ID[normalize_class_name(name)]: new_id
    for new_id, name in enumerate(CONSTRUCTION_NAMES)
}

print("Clases originales reales:")
for i, name in enumerate(ORIGINAL_NAMES):
    print(f"{i:02d} {name}")

print("\nClases filtradas para el proyecto:")
for i, name in enumerate(CONSTRUCTION_NAMES):
    print(f"{i:02d} {name}")

print("\nMapeo old->new:", OLD_TO_NEW)


In [ ]:
# ============================================================
# 6. Encontrar imágenes y etiquetas YOLO originales
# ============================================================

def find_file(root, name):
    matches = list(Path(root).rglob(name))
    return matches[0] if matches else None

def find_images(root):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sorted([p for p in Path(root).rglob("*") if p.suffix.lower() in exts])

def find_yolo_label_for_image(img_path, root):
    p = Path(img_path)
    candidates = []
    s = str(p)

    for old in ["images", "Images", "JPEGImages"]:
        if old in s:
            candidates.append(Path(s.replace(old, "labels")).with_suffix(".txt"))
            candidates.append(Path(s.replace(old, "Labels")).with_suffix(".txt"))

    candidates.append(p.with_suffix(".txt"))
    candidates += list(Path(root).rglob(p.stem + ".txt"))

    for c in candidates:
        if c.exists() and c.is_file():
            return c
    return None

all_images = find_images(RAW_DIR)
print("Total imágenes encontradas:", len(all_images))

if len(all_images) == 0:
    raise RuntimeError("No se encontraron imágenes. Revisa descarga de Kaggle.")

train_txt = find_file(RAW_DIR, "train_files.txt")
test_txt = find_file(RAW_DIR, "test_files.txt")

print("train_files.txt:", train_txt)
print("test_files.txt:", test_txt)

image_by_name = {p.name: p for p in all_images}
image_by_stem = {p.stem: p for p in all_images}

def resolve_list_file(txt_path):
    resolved = []
    if txt_path is None:
        return resolved
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    for line in lines:
        line = line.strip()
        if not line:
            continue
        candidate = Path(line)
        if candidate.exists():
            resolved.append(candidate.resolve())
        elif candidate.name in image_by_name:
            resolved.append(image_by_name[candidate.name].resolve())
        elif candidate.stem in image_by_stem:
            resolved.append(image_by_stem[candidate.stem].resolve())
    return sorted(set(resolved))

train_imgs_raw = resolve_list_file(train_txt)
test_imgs_raw = resolve_list_file(test_txt)

if len(train_imgs_raw) < 10:
    print("⚠️ No se pudo usar train_files.txt. Creando split aleatorio.")
    random.seed(42)
    imgs = all_images.copy()
    random.shuffle(imgs)
    n = len(imgs)
    train_imgs_raw = imgs[:int(n*0.8)]
    val_imgs_raw = imgs[int(n*0.8):int(n*0.9)]
    test_imgs_raw = imgs[int(n*0.9):]
else:
    if len(test_imgs_raw) == 0:
        remaining = [p for p in all_images if p not in set(train_imgs_raw)]
        if len(remaining) == 0:
            random.seed(42)
            train_imgs_raw = train_imgs_raw.copy()
            random.shuffle(train_imgs_raw)
            n_valtest = max(2, int(len(train_imgs_raw)*0.2))
            remaining = train_imgs_raw[-n_valtest:]
            train_imgs_raw = train_imgs_raw[:-n_valtest]
        random.seed(42)
        random.shuffle(remaining)
        half = max(1, len(remaining)//2)
        val_imgs_raw = remaining[:half]
        test_imgs_raw = remaining[half:]
    else:
        random.seed(42)
        random.shuffle(test_imgs_raw)
        half = max(1, len(test_imgs_raw)//2)
        val_imgs_raw = test_imgs_raw[:half]
        test_imgs_raw = test_imgs_raw[half:]

print("Split raw:")
print("Train:", len(train_imgs_raw), "Val:", len(val_imgs_raw), "Test:", len(test_imgs_raw))


In [ ]:
# ============================================================
# 7. Crear dataset filtrado para construcción
# ============================================================

def symlink_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def filter_label_file(src_label, dst_label):
    kept = []
    if src_label is None or not Path(src_label).exists():
        return 0

    for line in Path(src_label).read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            old_cls = int(float(parts[0]))
        except Exception:
            continue
        if old_cls not in OLD_TO_NEW:
            continue
        new_cls = OLD_TO_NEW[old_cls]
        kept.append(" ".join([str(new_cls)] + parts[1:5]))

    if kept:
        dst_label.parent.mkdir(parents=True, exist_ok=True)
        dst_label.write_text("\n".join(kept) + "\n", encoding="utf-8")
    return len(kept)

def create_filtered_split(raw_imgs, split_name):
    out_images = FILTERED_DIR / split_name / "images"
    out_labels = FILTERED_DIR / split_name / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    kept_images = []
    kept_objects = 0

    for img_path in raw_imgs:
        src_label = find_yolo_label_for_image(img_path, RAW_DIR)
        dst_img = out_images / img_path.name
        dst_label = out_labels / (img_path.stem + ".txt")

        n = filter_label_file(src_label, dst_label)
        if n > 0:
            symlink_or_copy(img_path, dst_img)
            kept_images.append(dst_img)
            kept_objects += n
        else:
            if dst_label.exists():
                dst_label.unlink()

    return kept_images, kept_objects

if FILTERED_DIR.exists():
    shutil.rmtree(FILTERED_DIR)
FILTERED_DIR.mkdir(parents=True, exist_ok=True)

train_imgs, train_objs = create_filtered_split(train_imgs_raw, "train")
val_imgs, val_objs = create_filtered_split(val_imgs_raw, "valid")
test_imgs, test_objs = create_filtered_split(test_imgs_raw, "test")

print("✅ Dataset filtrado creado")
print("Train:", len(train_imgs), "imágenes |", train_objs, "objetos")
print("Val:", len(val_imgs), "imágenes |", val_objs, "objetos")
print("Test:", len(test_imgs), "imágenes |", test_objs, "objetos")

if len(train_imgs) == 0 or len(val_imgs) == 0:
    raise RuntimeError("El filtrado dejó train/val vacío. Revisa estructura de etiquetas del dataset.")

data_yaml_path = FILTERED_DIR / "sh17_construction_data.yaml"
data_yaml = {
    "path": str(FILTERED_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CONSTRUCTION_NAMES),
    "names": CONSTRUCTION_NAMES,
}

with open(data_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("✅ YAML creado:", data_yaml_path)
print(data_yaml_path.read_text())


In [ ]:
# ============================================================
# 8. Distribución de clases filtradas
# ============================================================

def count_classes_in_split(split):
    label_dir = FILTERED_DIR / split / "labels"
    counts = {i: 0 for i in range(len(CONSTRUCTION_NAMES))}
    for txt in label_dir.glob("*.txt"):
        for line in txt.read_text(errors="ignore").splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cls = int(float(parts[0]))
                if cls in counts:
                    counts[cls] += 1
    return counts

for split in ["train", "valid", "test"]:
    counts = count_classes_in_split(split)
    print(f"\n{split.upper()}")
    for i, name in enumerate(CONSTRUCTION_NAMES):
        print(f"{i:02d} {name:12s}: {counts[i]}")

train_counts = count_classes_in_split("train")
plt.figure(figsize=(10,5))
plt.bar([CONSTRUCTION_NAMES[i] for i in train_counts.keys()], list(train_counts.values()))
plt.xticks(rotation=45, ha="right")
plt.title("Distribución de clases - Train filtrado construcción")
plt.ylabel("Instancias")
plt.tight_layout()
dist_path = FIG_DIR / "class_distribution_construction.png"
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura guardada:", dist_path)


In [ ]:
# ============================================================
# 9. Visualizar muestras del dataset filtrado
# ============================================================

sample_imgs = random.sample(train_imgs, min(8, len(train_imgs)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    img = cv2.imread(str(img_path))
    if img is None:
        ax.axis("off")
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = FILTERED_DIR / "train" / "labels" / (img_path.stem + ".txt")

    if lbl.exists():
        for line in lbl.read_text(errors="ignore").splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cls, cx, cy, bw, bh = map(float, parts[:5])
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                cls = int(cls)
                name = CONSTRUCTION_NAMES[cls]
                cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
                cv2.putText(img, name, (x1, max(15,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(img_path.name[:20], fontsize=8)

plt.tight_layout()
fig_path = FIG_DIR / "muestras_sh17_construction.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura guardada:", fig_path)


In [ ]:

# ============================================================
# 10. Baseline clasico HOG + SVM
# ============================================================

PPE_CLASS_NAMES = {"Glasses", "Gloves", "Shoes", "Safety-vest", "Helmet"}
PPE_NEW_IDS = {idx for idx, name in enumerate(CONSTRUCTION_NAMES) if name in PPE_CLASS_NAMES}
print("IDs EPP usados por baseline:", {CONSTRUCTION_NAMES[i]: i for i in sorted(PPE_NEW_IDS)})

def extract_hog_features(img_path, size=(128, 128)):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, size)
    return hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), feature_vector=True)

def binary_label_from_filtered(img_path, split):
    lbl = FILTERED_DIR / split / "labels" / (Path(img_path).stem + ".txt")
    if not lbl.exists():
        return None
    classes = []
    for line in lbl.read_text(errors="ignore").splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            classes.append(int(float(parts[0])))
    if not classes:
        return None
    return 1 if any(c in PPE_NEW_IDS for c in classes) else 0

def build_baseline_data(imgs, split, max_samples):
    X, y = [], []
    sample_imgs = imgs.copy()
    random.seed(42)
    random.shuffle(sample_imgs)
    for img_path in sample_imgs[:max_samples]:
        feat = extract_hog_features(img_path)
        label = binary_label_from_filtered(img_path, split)
        if feat is not None and label is not None:
            X.append(feat)
            y.append(label)
    return np.array(X), np.array(y)

print("Construyendo baseline HOG+SVM...")
X_train, y_train = build_baseline_data(train_imgs, "train", 1500)
X_val, y_val = build_baseline_data(val_imgs, "valid", 500)

baseline_acc = None
if len(X_train) < 20 or len(X_val) < 10 or len(set(y_train)) < 2 or len(set(y_val)) < 2:
    print("⚠️ No hay suficientes datos balanceados para baseline clasico.")
else:
    start = time.time()
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", LinearSVC(max_iter=5000, class_weight="balanced")),
    ])
    clf.fit(X_train, y_train)
    pred = clf.predict(X_val)
    train_time = time.time() - start
    baseline_acc = accuracy_score(y_val, pred)

    print("=== BASELINE HOG + SVM ===")
    print(f"Accuracy: {baseline_acc:.2%}")
    print(f"Tiempo entrenamiento: {train_time:.1f}s")
    print(classification_report(y_val, pred, target_names=["Sin EPP visible", "Con EPP visible"]))

    cm = confusion_matrix(y_val, pred)
    plt.figure(figsize=(5,4))
    plt.imshow(cm, cmap="Blues")
    plt.title("Matriz de confusion - Baseline")
    plt.colorbar()
    plt.xticks([0,1], ["Sin EPP", "Con EPP"])
    plt.yticks([0,1], ["Sin EPP", "Con EPP"])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i,j], ha="center", va="center", color="black")
    plt.xlabel("Predicho")
    plt.ylabel("Real")
    cm_path = FIG_DIR / "baseline_confusion_matrix.png"
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Figura guardada:", cm_path)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODEL_SIZE = "yolov8s.pt"  # probar yolov8m.pt si T4 con 16GB
EXPERIMENT_NAME = "sh17_construction_yolov8s_v2"
EPOCHS = 80  # ↑ 50→80 para convergencia con clases desbalanceadas
IMG_SIZE = 640
BATCH_SIZE = 16  # T4: 16; si OOM probar 8
RESUME_TRAINING = False

# --- Balanceo: oversampling de minoritarias en train ---
from collections import Counter

def _count_train():
    from pathlib import Path as _P
    c = Counter()
    for txt in (FILTERED_DIR / "train" / "labels").glob("*.txt"):
        for line in txt.read_text(errors="ignore").splitlines():
            parts = line.strip().split()
            if len(parts) >=5:
                c[int(float(parts[0]))] += 1
    return c

try:
    _cnt = _count_train()
    print("Distribución train pre-oversampling:")
    for i,n in enumerate(CONSTRUCTION_NAMES):
        print(f"{i:02d} {n:12s}: {_cnt.get(i,0)}")
    # Oversampling: duplicar imágenes con Safety-vest(9), Glasses(2), Ear(1), Shoes(8) (minoritarias)
    minority_ids = {1,2,8,9}  # Ear, Glasses, Shoes, Safety-vest
    train_labels = list((FILTERED_DIR / "train" / "labels").glob("*.txt"))
    minority_imgs = []
    for lbl in train_labels:
        ids = {int(float(p.split()[0])) for p in lbl.read_text(errors="ignore").splitlines() if len(p.split())>=5}
        if ids & minority_ids:
            minority_imgs.append(lbl.stem)
    print(f"Imágenes con minoritarias: {len(minority_imgs)} -> duplicando con augment copy-paste")
    import shutil as _sh
    for stem in minority_imgs:
        src_img = FILTERED_DIR / "train" / "images" / f"{stem}.jpg"
        # buscar ext real
        if not src_img.exists():
            for ext in [".jpg",".jpeg",".png",".webp"]:
                cand = FILTERED_DIR / "train" / "images" / f"{stem}{ext}"
                if cand.exists():
                    src_img = cand
                    break
        src_lbl = FILTERED_DIR / "train" / "labels" / f"{stem}.txt"
        if src_img.exists():
            dst_img = FILTERED_DIR / "train" / "images" / f"{stem}_os.jpg"
            dst_lbl = FILTERED_DIR / "train" / "labels" / f"{stem}_os.txt"
            if not dst_img.exists():
                _sh.copy2(src_img, dst_img)
                _sh.copy2(src_lbl, dst_lbl)
    _cnt2 = _count_train()
    print("Post-oversampling Safety-vest:", _cnt2.get(9,0), "Glasses:", _cnt2.get(2,0))
except Exception as e:
    print("Oversampling skip:", e)

last_pt = RUNS_DIR / EXPERIMENT_NAME / "weights" / "last.pt"

if RESUME_TRAINING and last_pt.exists():
    print("⚡ Retomando desde last.pt...")
    model = YOLO(str(last_pt))
    results = model.train(resume=True)
else:
    print("🚀 Entrenamiento mejorado desde cero (copy_paste 0.3, mosaic 1.0, AdamW 0.002, cls 0.6)...")
    model = YOLO(MODEL_SIZE)
    results = model.train(
        data=str(data_yaml_path),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        optimizer="AdamW",
        lr0=0.002,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        box=7.5,
        cls=0.6,
        dfl=1.5,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        shear=2.0,
        perspective=0.0001,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.3,
        copy_paste_mode="mixup",
        auto_augment="randaugment",
        erasing=0.2,
        cos_lr=True,
        close_mosaic=15,
        patience=25,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=EXPERIMENT_NAME,
        exist_ok=True,
        plots=True,
        save=True,
        verbose=True,
    )

print("✅ Entrenamiento finalizado")


In [ ]:
# Celda mantenida solo por compatibilidad con versiones anteriores del notebook.
# No descargues last.pt: el flujo final usa weights/best.pt generado por Ultralytics.
print("Usa el ZIP final o weights/best.pt; no se descarga last.pt.")


In [ ]:
# Celda mantenida solo por compatibilidad con versiones anteriores del notebook.
# No descargues last.pt: el flujo final usa weights/best.pt generado por Ultralytics.
print("Usa el ZIP final o weights/best.pt; no se descarga last.pt.")


In [ ]:

# ============================================================
# 11. Confirmar pesos finales
# ============================================================

best_model_path = RUNS_DIR / EXPERIMENT_NAME / "weights" / "best.pt"
last_model_path = RUNS_DIR / EXPERIMENT_NAME / "weights" / "last.pt"

if not best_model_path.exists():
    raise FileNotFoundError(f"No se encontro best.pt generado por Ultralytics: {best_model_path}")

print("✅ best.pt listo:", best_model_path)
if last_model_path.exists():
    print("ℹ️ last.pt tambien existe, pero NO se copia encima de best.pt:", last_model_path)


In [ ]:
# ============================================================
# 12. Evaluar modelo final
# ============================================================

best_model_path = RUNS_DIR / EXPERIMENT_NAME / "weights" / "best.pt"

if not best_model_path.exists():
    raise FileNotFoundError(f"No se encontró best.pt: {best_model_path}")

model_final = YOLO(str(best_model_path))

val_results = model_final.val(
    data=str(data_yaml_path),
    imgsz=IMG_SIZE,
    device=DEVICE,
    plots=True,
    verbose=True,
    save_json=True,
    save_hybrid=False
)

# Reporte por clase
try:
    import pandas as pd
    # box.maps por clase (si disponible)
    maps = getattr(val_results.box, 'maps', None)
    if maps is not None:
        for i, name in enumerate(CONSTRUCTION_NAMES):
            print(f"{name:12s} mAP@50: {maps[i]:.3f}")
except Exception as e:
    print('per-class skip', e)

map50 = float(val_results.box.map50)
map50_95 = float(val_results.box.map)
precision = float(val_results.box.mp)
recall = float(val_results.box.mr)
f1 = float((2 * precision * recall) / (precision + recall + 1e-9))

print("=== RESULTADOS FINALES YOLOv8 - SH17 FILTRADO CONSTRUCCIÓN ===")
print(f"mAP@50:    {map50:.3f} ({map50*100:.1f}%)")
print(f"mAP@50-95: {map50_95:.3f} ({map50_95*100:.1f}%)")
print(f"Precisión: {precision:.3f} ({precision*100:.1f}%)")
print(f"Recall:    {recall:.3f} ({recall*100:.1f}%)")
print(f"F1-Score:  {f1:.3f} ({f1*100:.1f}%)")


In [ ]:
# ============================================================
# 13. Curvas y detecciones de ejemplo
# ============================================================

run_dir = RUNS_DIR / EXPERIMENT_NAME
results_csv = run_dir / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    epochs_range = range(1, len(df)+1)

    plt.figure(figsize=(12,8))

    plt.subplot(2,2,1)
    for col in ["train/box_loss", "train/cls_loss", "train/dfl_loss"]:
        if col in df.columns:
            plt.plot(epochs_range, df[col], label=col)
    plt.title("Pérdidas de entrenamiento")
    plt.legend(); plt.grid(True)

    plt.subplot(2,2,2)
    for col in ["val/box_loss", "val/cls_loss", "val/dfl_loss"]:
        if col in df.columns:
            plt.plot(epochs_range, df[col], label=col)
    plt.title("Pérdidas de validación")
    plt.legend(); plt.grid(True)

    plt.subplot(2,2,3)
    map_cols = [c for c in df.columns if "mAP50" in c and "95" not in c]
    if map_cols:
        plt.plot(epochs_range, df[map_cols[0]], label="mAP@50")
    plt.title("mAP@50")
    plt.ylim(0,1); plt.legend(); plt.grid(True)

    plt.subplot(2,2,4)
    pcols = [c for c in df.columns if "precision" in c.lower()]
    rcols = [c for c in df.columns if "recall" in c.lower()]
    if pcols:
        plt.plot(epochs_range, df[pcols[0]], label="Precision")
    if rcols:
        plt.plot(epochs_range, df[rcols[0]], label="Recall")
    plt.title("Precision / Recall")
    plt.ylim(0,1); plt.legend(); plt.grid(True)

    plt.tight_layout()
    curves_path = FIG_DIR / "training_curves.png"
    plt.savefig(curves_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Curvas guardadas:", curves_path)

sample_pred_imgs = random.sample(val_imgs, min(9, len(val_imgs)))
pred_dir = OUT_DIR / "predicciones"
pred_dir.mkdir(parents=True, exist_ok=True)

for img_path in sample_pred_imgs:
    model_final.predict(
        source=str(img_path),
        conf=0.25,
        imgsz=IMG_SIZE,
        save=True,
        project=str(pred_dir),
        name="examples",
        exist_ok=True,
        verbose=False
    )

example_dir = pred_dir / "examples"
pred_images = list(example_dir.glob("*.*"))

if pred_images:
    plt.figure(figsize=(15,10))
    for i, img_path in enumerate(pred_images[:9]):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(3,3,i+1)
        plt.imshow(img)
        plt.axis("off")
    plt.tight_layout()
    det_grid = FIG_DIR / "detecciones_ejemplos.png"
    plt.savefig(det_grid, dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Detecciones guardadas:", det_grid)


In [ ]:
# ============================================================
# 14. Exportar ONNX y crear ZIP final
# ============================================================

onnx_path = None
try:
    onnx_path = Path(model_final.export(format="onnx", imgsz=IMG_SIZE, opset=12))
    print("✅ ONNX exportado:", onnx_path)
except Exception as e:
    print("⚠️ No se pudo exportar ONNX. No es obligatorio.")
    print(e)

baseline_txt = f"{baseline_acc:.2%}" if baseline_acc is not None else "No disponible"

summary_txt = BASE_DIR / "RESUMEN_FINAL.txt"
summary = f"""
RESUMEN FINAL DEL PROYECTO - SH17 FILTRADO PARA CONSTRUCCIÓN
==================================================

Proyecto:
Sistema de visión artificial para detectar trabajadores y verificar uso de EPP con YOLOv8.

Dataset base:
SH17 Dataset for PPE Detection.

Adaptación:
El dataset original contiene 17 clases de contextos industriales diversos.
Se filtraron las clases para dejar solo elementos relevantes para construcción.

Clases finales:
{CONSTRUCTION_NAMES}

Baseline:
HOG + SVM
Accuracy: {baseline_txt}

Modelo final:
YOLOv8s

Resultados YOLOv8:
mAP@50:    {map50:.3f} ({map50*100:.1f}%)
mAP@50-95: {map50_95:.3f} ({map50_95*100:.1f}%)
Precisión: {precision:.3f} ({precision*100:.1f}%)
Recall:    {recall:.3f} ({recall*100:.1f}%)
F1-Score:  {f1:.3f} ({f1*100:.1f}%)

Archivos:
weights/best.pt
weights/best.onnx si se generó

Uso en laptop:
Copiar weights/best.pt al proyecto:
epp_sh17_construction_project/weights/best.pt

Ejecutar:
python src/realtime_webcam.py --model weights/best.pt --camera 0 --conf 0.25

Limitación:
La alerta de incumplimiento en la demo se realiza a nivel de frame.
No asocia todavía cada elemento EPP con una persona específica.
""".strip()

summary_txt.write_text(summary, encoding="utf-8")
print(summary)

zip_path = "/content/epp_sh17_construction_resultados.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(str(best_model_path), "weights/best.pt")
    if onnx_path is not None and Path(onnx_path).exists():
        zf.write(str(onnx_path), "weights/best.onnx")

    zf.write(str(data_yaml_path), "dataset/sh17_construction_data.yaml")
    zf.write(str(summary_txt), "RESUMEN_FINAL.txt")

    if results_csv.exists():
        zf.write(str(results_csv), "results/results.csv")

    for fig in FIG_DIR.glob("*.*"):
        zf.write(str(fig), f"figures/{fig.name}")

    for fig_name in [
        "results.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "P_curve.png",
        "R_curve.png",
        "PR_curve.png",
        "labels.jpg",
        "train_batch0.jpg",
        "val_batch0_pred.jpg",
        "val_batch0_labels.jpg",
    ]:
        p = run_dir / fig_name
        if p.exists():
            zf.write(str(p), f"figures/{fig_name}")

print("✅ ZIP creado:", zip_path)

from google.colab import files
files.download(zip_path)
print("📥 Descarga iniciada")
